# Process raw data

Builds the analysis-ready files in `data/` from the `*_raw.csv` exports (git-ignored):

| raw | output |
|---|---|
| `participant_data_raw.csv` | `participant_data.csv` (wide, one row per participant) |
| `participant_data_long_raw.csv` | `participant_data_long.csv` (one row per participant x survey) |
| `participant_data_eod_raw.csv` | `participant_data_eod.csv` (one row per participant x end-of-day survey) |

Two things happen: Prolific IDs are replaced with a SHA-256 hash, and only columns read by the analyses in `analysis/` are kept
(item-level responses, submission timestamps, and raw conversation/usage logs are dropped).

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

DATA_DIR = Path(".")

In [2]:
def anonymize_pid(pid):
    return hashlib.sha256(str(pid).encode()).hexdigest()


def process(name, keep):
    raw = pd.read_csv(DATA_DIR / f"{name}_raw.csv", low_memory=False)
    raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]  # drop R/pandas row-index column
    missing = [c for c in keep if c not in raw.columns]
    assert not missing, f"{name}: columns not in raw data: {missing}"
    keep = set(keep)
    out = raw[[c for c in raw.columns if c in keep]].copy()  # preserve raw column order
    out["pid"] = out["pid"].map(anonymize_pid)
    out.to_csv(DATA_DIR / f"{name}.csv", index=False)
    print(f"{name}.csv: {out.shape[0]} rows, {raw.shape[1]} -> {out.shape[1]} columns")
    return out

## Wide participant data

In [3]:
# Outcomes measured at every survey (baseline, two diaries, midline, endline, post)
CORE_OUTCOMES = ["happy", "depressed", "anxious", "esi", "loneliness", "wb",
                 "BPNSFS_Autonomy", "BPNSFS_Relatedness", "BPNSFS_Competence"]
# Outcomes measured only in the full surveys
SOCIAL_OUTCOMES = ["lubben", "social_satisfaction", "time_spend_social", "ppl_met_inperson",
                   "problematic_use", "belief"]
# Per-survey design variables
DESIGN = ["is_weekend", "date_since_start", "wave"]

wide_cols = ["pid", "baseline_treatment"]
for tp in ["baseline", "diary1", "midline", "diary2", "endline", "post"]:
    wide_cols += [f"{tp}_{v}" for v in CORE_OUTCOMES + DESIGN]
for tp in ["baseline", "midline", "endline"]:
    wide_cols += [f"{tp}_{v}" for v in SOCIAL_OUTCOMES]

# Baseline demographics and usage intensity
wide_cols += [f"baseline_{v}" for v in ["Gender", "Age", "Race", "EdLevel", "Income", "Relationship", "intensity"]]

# Follow-up survey: self-report + exported usage metadata two weeks before/after the intervention
wide_cols += ["post_intensity", "post_Usage", "post_problematic_use", "post_belief"]
wide_cols += [f"post_{w}_{m}" for w in ["before", "after"]
              for m in ["totalMessages", "sessions", "timeSpentSeconds"]]

# Person-level covariates and study bookkeeping
wide_cols += ["Usage", "Productivity", "Entertainment", "Social", "Novelty", "Tenure",
              "isProductivityMain", "isSocialMain", "wave", "Total approvals",
              "completed_endline", "completed_post", "provide_metadata", "days_deactivated"]

wide = process("participant_data", wide_cols)

participant_data.csv: 625 rows, 594 -> 123 columns


## Long participant data

In [4]:
long_cols = [
    "pid", "wave", "treatment", "time", "is_weekend", "date_since_start", "isDeactivation",
    # person-level covariates
    "Tenure", "Usage", "Social", "Productivity", "Relationship", "Novelty", "Entertainment",
    "isProductivityMain", "isSocialMain",
    "baseline_Age", "baseline_Gender", "baseline_EdLevel", "baseline_Income", "baseline_Race",
    # outcomes
    *CORE_OUTCOMES, *SOCIAL_OUTCOMES,
    "baseline_lubben", "baseline_ppl_met_inperson", "baseline_social_satisfaction",
]

long = process("participant_data_long", long_cols)

participant_data_long.csv: 3125 rows, 66 -> 39 columns


## End-of-day participant data

In [5]:
eod_cols = [
    "pid", "day", "days_since_start", "is_weekend", "treatment", "wave", "isDeactivation",
    # outcomes
    "happy", "depressed", "anxious", "esi", "time_worked", "time_socializing", "log_num_interactions",
    # person-level covariates
    "baseline_Age", "baseline_Gender", "baseline_Race", "baseline_EdLevel", "baseline_Income",
    "Relationship", "Tenure", "Usage", "Social", "Productivity",
]

eod = process("participant_data_eod", eod_cols)

participant_data_eod.csv: 5541 rows, 26 -> 24 columns
